Binary Classification Using DistilBERT


In [ ]:
!pip install transformers datasets torch scikit-learn pandas

import pandas as pd
import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import classification_report, average_precision_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from google.colab import files

# ==========================================
# 1. LOAD AND PREPARE DATA
# ==========================================
print("Loading data...")
# Make sure your file paths are exactly correct
df_train = pd.read_csv('/content/drive/MyDrive/NLP PROJECT FILES/Binary classification Task/Binarylabel_Train.csv').dropna(subset=['text', 'label'])
df_val = pd.read_csv('/content/drive/MyDrive/NLP PROJECT FILES/Binary classification Task/Binarylabel_Val.csv').dropna(subset=['text', 'label'])
df_test = pd.read_excel('/content/drive/MyDrive/NLP PROJECT FILES/Binarylabel_Test.xlsx').dropna(subset=['text'])

# Convert to Hugging Face Dataset format
ds_train = Dataset.from_pandas(df_train[['text', 'label']])
ds_val = Dataset.from_pandas(df_val[['text', 'label']])
ds_test = Dataset.from_pandas(df_test[['text']]) # Test set has no labels for the model to see

# ==========================================
# 2. LOAD DISTILBERT TOKENIZER & MODEL
# ==========================================
model_name = "distilbert-base-uncased"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

print("Tokenizing datasets...")
tokenized_train = ds_train.map(tokenize_function, batched=True)
tokenized_val = ds_val.map(tokenize_function, batched=True)
tokenized_test = ds_test.map(tokenize_function, batched=True)

# ==========================================
# 3. HANDLE CLASS IMBALANCE (CUSTOM TRAINER)
# ==========================================
class_weights = compute_class_weight('balanced', classes=np.unique(df_train['label']), y=df_train['label'])
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Dynamically place weights on the exact device the model is currently using
        local_weights_tensor = weights_tensor.to(model.device)
        loss_fct = nn.CrossEntropyLoss(weight=local_weights_tensor)

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ==========================================
# 4. METRICS & TRAINING ARGUMENTS
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    predictions = np.argmax(logits, axis=-1)

    return {
        "pr_auc": average_precision_score(labels, probs),
        "f1_macro": f1_score(labels, predictions, average='macro')
    }

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",         # THE FIX: Evaluate every epoch
    save_strategy="epoch",         # THE FIX: Save every epoch to match eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir='./logs',
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc",
    save_total_limit=2             # THE FIX: Prevents Colab disk from filling up
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# ==========================================
# 5. TRAIN AND EVALUATE ON VALIDATION SET
# ==========================================
print("Starting DistilBERT Fine-Tuning...")
trainer.train()

print("\n--- Validation Results ---")
val_predictions = trainer.predict(tokenized_val)
val_probs = torch.nn.functional.softmax(torch.tensor(val_predictions.predictions), dim=-1)[:, 1].numpy()
val_preds = np.argmax(val_predictions.predictions, axis=-1)
val_labels = val_predictions.label_ids

print(f"Validation PR-AUC: {average_precision_score(val_labels, val_probs):.4f}")
print(f"Validation Macro F1: {f1_score(val_labels, val_preds, average='macro'):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(val_labels, val_preds))
print("\nClassification Report:")
print(classification_report(val_labels, val_preds))

# ==========================================
# 6. INFERENCE ON TEST SET & EXPORT
# ==========================================
print("\n--- Generating Final Predictions for Test Set ---")
test_predictions = trainer.predict(tokenized_test)
test_preds = np.argmax(test_predictions.predictions, axis=-1)

# Make sure we grab the correct ID column. If your column is named 'ID' or 'id', change 'unique_id' below.
id_column_name = 'unique_id' if 'unique_id' in df_test.columns else df_test.columns[0]

output_df = pd.DataFrame({
    'unique_id': df_test[id_column_name],
    'label': test_preds
})

output_filename = 'DistilBERT_Test_Predictions.csv'
output_df.to_csv(output_filename, index=False)
print(f"Successfully created {output_filename} with {len(output_df)} records.")

files.download(output_filename)